# LatentSpace Agentic Training Dataset Pipeline
### Offline Semantic Middleware & Agentic Assistant Dataset Builder (Revised Architecture)
**Pipeline Version:** 2.0.0  
**Target Assistant:** LatentSpace Local Semantic Middleware  
**Core Principle:** `original_dataset.csv` contains ONLY `raw_text` and `source` (exactly 23,384 rows).  
**Noise-Inclusive:** Retains real-world speech disfluencies, noisy transcripts, and OCR lines so the model learns error handling and clarification.  
**Strict True LLM Only:** Zero fake/heuristic annotations. Unannotated rows remain pending in `original_dataset.csv`.  
**Failed Annotations Isolation:** All LLM or validation errors are logged to `output/failed_annotations.csv`.  
**Universal Tool Contract:** `dispatch_actions`  
**Provider:** OpenRouter (`qwen/qwen3.8-27b:free`, strict free-only routing)  

```text
Multiple Source Datasets
        ↓
FETCH / EXTRACT ORIGINAL CONTENT
        ↓
ORIGINAL DATASET (raw_text + source ONLY, 23,384 rows)
        ↓
TRUE LLM AGENTIC ANNOTATION (OpenRouter API)
   ├── Success ──> VALIDATION ──> SYNTHETIC DATASET & TRAINING JSONL
   └── Failure / Error ─────────> FAILED ANNOTATIONS (failed_annotations.csv)
```


## 1. Installation
Install and verify required runtime libraries for dataset loading, text processing, and schema validation.

In [87]:
# 1. Installation verification
import sys
import os
import subprocess

required_pkgs = ["pandas", "datasets", "requests", "tqdm", "jsonschema"]
missing_pkgs = []
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f"Installing missing packages: {missing_pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_pkgs)
else:
    print("All required packages are installed.")


All required packages are installed.


## 2. Imports
Import standard library modules, data science tools, and dataset utilities.

In [88]:
# 2. Imports
import os
import sys
import json
import re
import time
import math
import hashlib
import random
import urllib.request
import requests
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Any, Tuple
from abc import ABC, abstractmethod

import pandas as pd
import numpy as np

print("Imports completed successfully.")


Imports completed successfully.


## 3. Configuration
Centralized immutable configuration for the LatentSpace dataset pipeline.

In [89]:
# 3. Pipeline Configuration
@dataclass(frozen=True)
class PipelineConfig:
    # ---------------------------------------------------------
    # Pipeline Identity
    # ---------------------------------------------------------
    pipeline_version: str = "2.0.0"
    random_seed: int = 42

    # ---------------------------------------------------------
    # LLM Provider
    # ---------------------------------------------------------
    llm_provider: str = "openrouter"
    api_base_url: str = "https://openrouter.ai/api/v1"

    annotation_model: str = "qwen/qwen3.8-27b:free"
    synthetic_model: str = "qwen/qwen3.8-27b:free"
    validation_model: str = "qwen/qwen3.8-27b:free"
    connectivity_test_model: str = "qwen/qwen3.8-27b:free"

    # Do not silently switch to another model/provider.
    allow_fallbacks: bool = False

    # ---------------------------------------------------------
    # API Request / Quota Controls
    # ---------------------------------------------------------
    daily_request_limit: int = 50
    stop_before_daily_limit: bool = True

    batch_size: int = 10
    max_batch_chars: int = 12000
    max_batch_tokens: int = 4096

    fallback_to_single_on_failure: bool = True
    max_retries: int = 3
    backoff_factor: float = 2.0
    request_timeout: int = 60

    # ---------------------------------------------------------
    # Requested Source Dataset Targets
    #
    # These are requested baseline counts, NOT guaranteed counts.
    # The pipeline must never duplicate records just to reach them.
    # Actual available/valid counts are reported separately.
    # ---------------------------------------------------------
    target_banking77: int = 7000
    target_clinc150: int = 8000
    target_hwu64: int = 5000

    # MINDS-14
    target_minds14_us: int = 563
    target_minds14_au: int = 654
    target_minds14_gb: int = 592
    target_minds14_ext: int = 1246

    # OCR / document datasets
    target_cord_v2: int = 800
    target_sroie: int = 626
    target_funsd: int = 149

    # Requested baseline total.
    # This is calculated from the targets above and is NOT
    # a guarantee of the final dataset size.
    total_target: int = 23384

    # ---------------------------------------------------------
    # Dataset Processing
    # ---------------------------------------------------------
    english_only: bool = True

    # Original dataset is immutable:
    # exactly raw_text + source.
    original_raw_text_column: str = "raw_text"
    original_source_column: str = "source"

    # Do not modify, translate, paraphrase, normalize, or correct
    # original source text.
    preserve_original_text: bool = True

    # Do not duplicate records to satisfy source targets.
    allow_target_padding: bool = False

    # ---------------------------------------------------------
    # Test / Development Modes
    # ---------------------------------------------------------
    dry_run_mode: bool = False
    test_mode_batch_limit: int = 5

    # ---------------------------------------------------------
    # Directory Structure
    # ---------------------------------------------------------
    base_dir: str = "latentspace_dataset"

    output_dir: str = "latentspace_dataset/output"
    cache_dir: str = "latentspace_dataset/cache"
    config_dir: str = "latentspace_dataset/config"
    prompts_dir: str = "latentspace_dataset/prompts"

    # Source/raw dataset artifacts
    source_dir: str = "latentspace_dataset/source"
    raw_dir: str = "latentspace_dataset/raw"

    # Validation/rejection artifacts
    validation_dir: str = "latentspace_dataset/validation"
    rejected_dir: str = "latentspace_dataset/rejected"


config = PipelineConfig()

calculated_target = (
    config.target_banking77
    + config.target_clinc150
    + config.target_hwu64
    + config.target_minds14_us
    + config.target_minds14_au
    + config.target_minds14_gb
    + config.target_cord_v2
    + config.target_sroie
    + config.target_funsd
)

print("Pipeline Config initialized.")
print(f"Pipeline version: {config.pipeline_version}")
print(f"LLM provider: {config.llm_provider}")
print(f"Annotation model: {config.annotation_model}")
print(f"Daily request limit: {config.daily_request_limit}")
print(f"Requested source target: {calculated_target:,} records")
print(f"Configured total target: {config.total_target:,}")

if calculated_target != config.total_target:
    raise ValueError(
        f"Target mismatch: individual targets sum to "
        f"{calculated_target:,}, but total_target is "
        f"{config.total_target:,}."
    )

print("Target configuration validated.")

Pipeline Config initialized. Target records: 23,384


## 4. Security / Environment Validation
Safely validate credentials without printing keys, and initialize the Request Ledger and Checkpoint Manager.

In [91]:
# 4. Security, Quota Ledger, and Checkpoint Manager

# ---------------------------------------------------------
# OpenRouter API Key
# ---------------------------------------------------------
api_key = os.environ.get("OPENROUTER_API_KEY", "")

try:
    from google.colab import userdata
    api_key = api_key or userdata.get("OPENROUTER_API_KEY", "")
except Exception:
    pass

if api_key:
    masked_key = (
        api_key[:4] + "..." + api_key[-4:]
        if len(api_key) > 8
        else "***"
    )
    print(f"OPENROUTER_API_KEY is configured: {masked_key}")
else:
    print(
        "OPENROUTER_API_KEY not found. "
        "The pipeline can still build the original dataset, "
        "but LLM annotation/synthesis will not run."
    )


# ---------------------------------------------------------
# Request Ledger
# ---------------------------------------------------------
class RequestLedger:
    """
    Tracks OpenRouter request attempts and local production quota.

    Important:
    - Every production HTTP attempt consumes one request.
    - Retries also consume one request.
    - Test requests do not consume production quota.
    - Cache hits do not consume quota.
    - API keys and raw user/source text are never written here.
    """

    def __init__(self, config: PipelineConfig):
        self.config = config

        self.ledger_path = os.path.join(
            config.cache_dir,
            "request_ledger.jsonl"
        )

        os.makedirs(
            os.path.dirname(self.ledger_path),
            exist_ok=True
        )

        self.production_requests_used = 0
        self.cache_hits = 0
        self.retries = 0

        self._load_state()

    def _load_state(self):
        """
        Restore today's production request count from the ledger.

        Only valid JSONL records are processed. Corrupted lines are
        ignored so one damaged ledger entry does not prevent recovery.
        """

        today = datetime.now(timezone.utc).strftime("%Y-%m-%d")

        if not os.path.exists(self.ledger_path):
            return

        with open(
            self.ledger_path,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:
                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)

                    if (
                        record.get("quota_date") == today
                        and record.get("is_test", False) is False
                    ):
                        self.production_requests_used += 1

                    if record.get("retry_number", 0) > 0:
                        self.retries += 1

                except (json.JSONDecodeError, TypeError):
                    continue

    def record_attempt(self, attempt_data: dict):
        """
        Record one HTTP request attempt.

        The caller should provide metadata such as:
        stage, model, request_hash, retry_number, HTTP status,
        latency, record counts, and error category.

        Sensitive information must never be included.
        """

        record = dict(attempt_data)

        today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        record["quota_date"] = today

        # Explicitly normalize test flag.
        is_test = bool(record.get("is_test", False))
        record["is_test"] = is_test

        # Every production attempt consumes one request.
        if not is_test:
            self.production_requests_used += 1

        # Track retries separately.
        if record.get("retry_number", 0) > 0:
            self.retries += 1

        # JSONL requires exactly one JSON object per line.
        with open(
            self.ledger_path,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True
                )
                + "\n"
            )

    def record_cache_hit(self):
        """
        Cache hits do not consume an OpenRouter request.
        """
        self.cache_hits += 1

    def can_request(self) -> bool:
        """
        Determine whether another production request may be made.
        """

        if not self.config.stop_before_daily_limit:
            return True

        return (
            self.production_requests_used
            < self.config.daily_request_limit
        )

    def remaining_requests(self) -> int:
        """
        Return remaining production requests for today.
        """

        return max(
            0,
            self.config.daily_request_limit
            - self.production_requests_used
        )

    def status(self) -> dict:
        """
        Return a safe summary suitable for logging/display.
        """

        return {
            "production_requests_used": self.production_requests_used,
            "daily_request_limit": self.config.daily_request_limit,
            "remaining_requests": self.remaining_requests(),
            "cache_hits": self.cache_hits,
            "retries": self.retries,
        }


# ---------------------------------------------------------
# Atomic Checkpoint Manager
# ---------------------------------------------------------
class CheckpointManager:
    """
    Provides crash-safe JSON checkpoint storage.

    Writes are performed to a temporary file first and then
    atomically replaced into the final location.
    """

    @staticmethod
    def atomic_write_json(
        file_path: str,
        data: Any
    ):
        os.makedirs(
            os.path.dirname(file_path),
            exist_ok=True
        )

        temp_path = (
            f"{file_path}.tmp."
            f"{os.getpid()}."
            f"{time.time_ns()}"
        )

        with open(
            temp_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )

            f.flush()
            os.fsync(f.fileno())

        os.replace(
            temp_path,
            file_path
        )

    @staticmethod
    def load_json(
        file_path: str
    ) -> Optional[Any]:

        if not os.path.exists(file_path):
            return None

        try:
            with open(
                file_path,
                "r",
                encoding="utf-8"
            ) as f:

                return json.load(f)

        except (
            json.JSONDecodeError,
            OSError,
            TypeError
        ):
            return None


# ---------------------------------------------------------
# Initialize Managers
# ---------------------------------------------------------
ledger = RequestLedger(config)

checkpoint_manager = CheckpointManager()

ledger_status = ledger.status()

print(
    "RequestLedger initialized."
)

print(
    f"Production requests used today: "
    f"{ledger_status['production_requests_used']}/"
    f"{ledger_status['daily_request_limit']}"
)

print(
    f"Remaining production requests: "
    f"{ledger_status['remaining_requests']}"
)

print(
    f"Recorded cache hits: "
    f"{ledger_status['cache_hits']}"
)

print(
    f"Recorded retries: "
    f"{ledger_status['retries']}"
)

OPENROUTER_API_KEY is configured: sk-o...2c74
RequestLedger initialized. Production requests used today: 0/50


## 5. Controlled Domain & Category Taxonomy
Controlled taxonomy and universal tool contract for agentic routing.

In [ ]:
# 5. Controlled Taxonomy & Universal Tool Schema

# ---------------------------------------------------------
# Controlled Domains
# ---------------------------------------------------------
DOMAINS = [
    "finance",
    "productivity",
    "travel",
    "commerce",
    "communication",
    "account_and_security",
    "documents",
    "information",
    "other",
]


# ---------------------------------------------------------
# Controlled Categories
# ---------------------------------------------------------
CATEGORIES = [
    # Finance
    "finance/account",
    "finance/transactions",
    "finance/payments",
    "finance/transfers",
    "finance/cards",
    "finance/expenses",

    # Productivity
    "productivity/tasks",
    "productivity/reminders",
    "productivity/alarms",
    "productivity/notes",
    "productivity/calendar",

    # Travel
    "travel/flights",
    "travel/reservations",
    "travel/transportation",

    # Commerce
    "commerce/products",
    "commerce/orders",
    "commerce/subscriptions",
    "commerce/purchases",

    # Communication
    "communication/messages",
    "communication/email",
    "communication/contacts",

    # Account & Security
    "account_and_security/profile",
    "account_and_security/authentication",
    "account_and_security/credentials",
    "account_and_security/security",
    "account_and_security/cards",

    # Documents
    "documents/ocr",
    "documents/extraction",
    "documents/receipts",
    "documents/invoices",
    "documents/forms",

    # Information
    "information/general_information",
    "information/navigation",
    "information/calculation",

    # Other
    "other/clarification",
    "other/out_of_scope",
]


# ---------------------------------------------------------
# Universal Dispatch Tool
#
# IMPORTANT:
# This schema describes actions that the assistant intends
# to route to the downstream application.
#
# The LLM does NOT execute these actions itself.
# ---------------------------------------------------------
UNIVERSAL_DISPATCH_TOOL = {
    "type": "function",
    "function": {
        "name": "dispatch_actions",

        "description": (
            "Routes one or more structured actions to the "
            "downstream system based on the user's request. "
            "Actions may be independent or may depend on "
            "previous actions."
        ),

        "parameters": {
            "type": "object",

            "properties": {
                "actions": {
                    "type": "array",

                    "description": (
                        "List of structured actions to route "
                        "to the downstream system."
                    ),

                    "items": {
                        "type": "object",

                        "properties": {
                            "action_id": {
                                "type": "string",

                                "description": (
                                    "Unique identifier for this "
                                    "action within the current "
                                    "dispatch, such as a1 or a2."
                                ),

                                "pattern": "^a[0-9]+$",
                            },

                            "intent_name": {
                                "type": "string",

                                "description": (
                                    "Intent name from the strict "
                                    "registered intent taxonomy. "
                                    "Arbitrary intent names are "
                                    "not permitted."
                                ),
                            },

                            "depends_on": {
                                "type": "array",

                                "description": (
                                    "Action IDs that must be "
                                    "completed before this action "
                                    "can proceed. Use an empty "
                                    "array when there is no "
                                    "dependency."
                                ),

                                "items": {
                                    "type": "string",
                                    "pattern": "^a[0-9]+$",
                                },

                                "default": [],
                            },

                            "entities": {
                                "type": "object",

                                "description": (
                                    "Structured entities and "
                                    "parameters explicitly "
                                    "supported by the intent."
                                ),

                                "additionalProperties": True,
                            },
                        },

                        "required": [
                            "action_id",
                            "intent_name",
                            "entities",
                        ],

                        "additionalProperties": False,
                    },
                },
            },

            "required": [
                "actions",
            ],

            "additionalProperties": False,
        },
    },
}


# ---------------------------------------------------------
# Taxonomy Validation Helpers
# ---------------------------------------------------------
def validate_domain(domain: str) -> bool:
    """Return True when the domain belongs to the controlled taxonomy."""
    return domain in DOMAINS


def validate_category(category: str) -> bool:
    """Return True when the category belongs to the controlled taxonomy."""
    return category in CATEGORIES


def category_matches_domain(
    domain: str,
    category: str
) -> bool:
    """
    Verify that a category belongs to the specified domain.

    Example:
        finance + finance/transfers -> True
        finance + productivity/tasks -> False
    """

    if not validate_domain(domain):
        return False

    if not validate_category(category):
        return False

    return category.startswith(f"{domain}/")


# ---------------------------------------------------------
# Taxonomy Summary
# ---------------------------------------------------------
print(
    f"Taxonomy defined: "
    f"{len(DOMAINS)} domains, "
    f"{len(CATEGORIES)} categories."
)

print(
    f"Universal tool: "
    f"{UNIVERSAL_DISPATCH_TOOL['function']['name']}"
)

print("Taxonomy validation helpers initialized.")

## 6. Intent Registry
The strict source of truth for valid LatentSpace intents.

In [ ]:
# 6. Strict Intent Registry

# ---------------------------------------------------------
# Strict Intent Registry
#
# This registry is the ONLY approved source of executable
# intent names for the agentic training dataset.
#
# The LLM must NEVER invent arbitrary intent names.
# ---------------------------------------------------------

DEFAULT_INTENT_REGISTRY = {
    "intents": {

        # =================================================
        # FINANCE / BANKING
        # =================================================

        "log_expense": {
            "domain": "finance",
            "category": "finance/expenses",
            "description": "Record a new expense.",
            "requires_entities": ["amount", "description"],
            "optional_entities": [
                "category",
                "payment_type",
                "date",
                "merchant"
            ]
        },

        "delete_expense": {
            "domain": "finance",
            "category": "finance/expenses",
            "description": "Delete an existing expense record.",
            "requires_entities": [],
            "optional_entities": [
                "expense_id",
                "description",
                "amount",
                "date"
            ]
        },

        "search_expense": {
            "domain": "finance",
            "category": "finance/expenses",
            "description": "Find or retrieve existing expense records.",
            "requires_entities": [],
            "optional_entities": [
                "description",
                "category",
                "amount",
                "date",
                "date_range"
            ]
        },

        "update_expense": {
            "domain": "finance",
            "category": "finance/expenses",
            "description": "Modify an existing expense record.",
            "requires_entities": [],
            "optional_entities": [
                "expense_id",
                "description",
                "amount",
                "category",
                "payment_type",
                "date"
            ]
        },

        "transfer_money": {
            "domain": "finance",
            "category": "finance/transfers",
            "description": "Transfer money between supported accounts.",
            "requires_entities": ["amount"],
            "optional_entities": [
                "source_account",
                "destination_account",
                "recipient",
                "date"
            ]
        },

        "check_balance": {
            "domain": "finance",
            "category": "finance/account",
            "description": "Check an account balance.",
            "requires_entities": [],
            "optional_entities": [
                "account"
            ]
        },

        "check_transaction": {
            "domain": "finance",
            "category": "finance/transactions",
            "description": "Retrieve or inspect a financial transaction.",
            "requires_entities": [],
            "optional_entities": [
                "transaction_id",
                "description",
                "amount",
                "date",
                "merchant"
            ]
        },

        "cancel_transfer": {
            "domain": "finance",
            "category": "finance/transfers",
            "description": "Cancel an eligible money transfer.",
            "requires_entities": [],
            "optional_entities": [
                "transfer_id",
                "recipient",
                "amount",
                "date"
            ]
        },

        "card_lost": {
            "domain": "finance",
            "category": "finance/cards",
            "description": "Report a lost payment card.",
            "requires_entities": [],
            "optional_entities": [
                "card_type",
                "last_four"
            ]
        },

        "card_stolen": {
            "domain": "finance",
            "category": "finance/cards",
            "description": "Report a stolen payment card.",
            "requires_entities": [],
            "optional_entities": [
                "card_type",
                "last_four"
            ]
        },

        "freeze_card": {
            "domain": "finance",
            "category": "finance/cards",
            "description": "Freeze or temporarily block a payment card.",
            "requires_entities": [],
            "optional_entities": [
                "card_type",
                "last_four"
            ]
        },

        "unfreeze_card": {
            "domain": "finance",
            "category": "finance/cards",
            "description": "Unfreeze or unblock a payment card.",
            "requires_entities": [],
            "optional_entities": [
                "card_type",
                "last_four"
            ]
        },

        # =================================================
        # PRODUCTIVITY
        # =================================================

        "create_task": {
            "domain": "productivity",
            "category": "productivity/tasks",
            "description": "Create a task.",
            "requires_entities": ["task"],
            "optional_entities": [
                "due_date",
                "priority",
                "category"
            ]
        },

        "complete_task": {
            "domain": "productivity",
            "category": "productivity/tasks",
            "description": "Mark an existing task as completed.",
            "requires_entities": [],
            "optional_entities": [
                "task_id",
                "task"
            ]
        },

        "delete_task": {
            "domain": "productivity",
            "category": "productivity/tasks",
            "description": "Delete an existing task.",
            "requires_entities": [],
            "optional_entities": [
                "task_id",
                "task"
            ]
        },

        "update_task": {
            "domain": "productivity",
            "category": "productivity/tasks",
            "description": "Modify an existing task.",
            "requires_entities": [],
            "optional_entities": [
                "task_id",
                "task",
                "due_date",
                "priority",
                "category"
            ]
        },

        "search_task": {
            "domain": "productivity",
            "category": "productivity/tasks",
            "description": "Find or retrieve existing tasks.",
            "requires_entities": [],
            "optional_entities": [
                "task",
                "status",
                "due_date",
                "category"
            ]
        },

        "create_reminder": {
            "domain": "productivity",
            "category": "productivity/reminders",
            "description": "Create a reminder.",
            "requires_entities": ["reminder"],
            "optional_entities": [
                "date",
                "time",
                "datetime"
            ]
        },

        "cancel_reminder": {
            "domain": "productivity",
            "category": "productivity/reminders",
            "description": "Cancel an existing reminder.",
            "requires_entities": [],
            "optional_entities": [
                "reminder_id",
                "reminder"
            ]
        },

        "update_reminder": {
            "domain": "productivity",
            "category": "productivity/reminders",
            "description": "Modify an existing reminder.",
            "requires_entities": [],
            "optional_entities": [
                "reminder_id",
                "reminder",
                "date",
                "time",
                "datetime"
            ]
        },

        "search_reminder": {
            "domain": "productivity",
            "category": "productivity/reminders",
            "description": "Find or retrieve existing reminders.",
            "requires_entities": [],
            "optional_entities": [
                "reminder",
                "date",
                "status"
            ]
        },

        "create_alarm": {
            "domain": "productivity",
            "category": "productivity/alarms",
            "description": "Create an alarm.",
            "requires_entities": ["time"],
            "optional_entities": [
                "label",
                "repeat"
            ]
        },

        "cancel_alarm": {
            "domain": "productivity",
            "category": "productivity/alarms",
            "description": "Cancel an existing alarm.",
            "requires_entities": [],
            "optional_entities": [
                "alarm_id",
                "label",
                "time"
            ]
        },

        "update_alarm": {
            "domain": "productivity",
            "category": "productivity/alarms",
            "description": "Modify an existing alarm.",
            "requires_entities": [],
            "optional_entities": [
                "alarm_id",
                "time",
                "label",
                "repeat"
            ]
        },

        "search_alarm": {
            "domain": "productivity",
            "category": "productivity/alarms",
            "description": "Find or retrieve existing alarms.",
            "requires_entities": [],
            "optional_entities": [
                "alarm_id",
                "label",
                "time"
            ]
        },

        # =================================================
        # TRAVEL / BOOKING
        # =================================================

        "book_reservation": {
            "domain": "travel",
            "category": "travel/reservations",
            "description": "Create a reservation.",
            "requires_entities": [],
            "optional_entities": [
                "venue",
                "date",
                "time",
                "party_size"
            ]
        },

        "cancel_reservation": {
            "domain": "travel",
            "category": "travel/reservations",
            "description": "Cancel an existing reservation.",
            "requires_entities": [],
            "optional_entities": [
                "reservation_id",
                "venue",
                "date"
            ]
        },

        "search_reservation": {
            "domain": "travel",
            "category": "travel/reservations",
            "description": "Find an existing reservation.",
            "requires_entities": [],
            "optional_entities": [
                "reservation_id",
                "venue",
                "date"
            ]
        },

        "search_flight": {
            "domain": "travel",
            "category": "travel/flights",
            "description": "Search for available flights.",
            "requires_entities": [],
            "optional_entities": [
                "origin",
                "destination",
                "date",
                "passengers"
            ]
        },

        "book_flight": {
            "domain": "travel",
            "category": "travel/flights",
            "description": "Book a flight.",
            "requires_entities": [],
            "optional_entities": [
                "origin",
                "destination",
                "date",
                "passengers"
            ]
        },

        "cancel_flight": {
            "domain": "travel",
            "category": "travel/flights",
            "description": "Cancel an existing flight booking.",
            "requires_entities": [],
            "optional_entities": [
                "booking_id",
                "flight",
                "date"
            ]
        },

        # =================================================
        # GENERAL / CONTROL
        # =================================================

        "search_information": {
            "domain": "information",
            "category": "information/general_information",
            "description": "Retrieve general information.",
            "requires_entities": ["query"],
            "optional_entities": []
        },

        "clarification_required": {
            "domain": "other",
            "category": "other/clarification",
            "description": (
                "Use when the user's intended action cannot "
                "be determined reliably from the available context."
            ),
            "requires_entities": [],
            "optional_entities": [
                "missing_information"
            ]
        },

        "out_of_scope": {
            "domain": "other",
            "category": "other/out_of_scope",
            "description": (
                "Use when the request cannot be handled by "
                "the supported intent registry."
            ),
            "requires_entities": [],
            "optional_entities": []
        },
    }
}


# ---------------------------------------------------------
# Intent Registry Class
# ---------------------------------------------------------
class IntentRegistry:

    def __init__(
        self,
        config_path: str,
        default_registry: Optional[dict] = None
    ):
        self.config_path = config_path
        self.intents: Dict[str, dict] = {}

        self.load(default_registry)

    def load(
        self,
        default_registry: Optional[dict] = None
    ):
        """
        Load the registry from disk when available.

        If no file exists, use the controlled registry defined
        in this notebook.

        The registry must never silently become empty.
        """

        if os.path.exists(self.config_path):

            with open(
                self.config_path,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

            loaded = data.get("intents", data)

            if not isinstance(loaded, dict) or not loaded:
                raise ValueError(
                    "Intent registry file exists but contains "
                    "no valid intents."
                )

            self.intents = loaded

        else:

            if default_registry is None:
                raise FileNotFoundError(
                    f"Intent registry not found: {self.config_path}"
                )

            self.intents = default_registry.get(
                "intents",
                default_registry
            )

    def is_valid_intent(
        self,
        intent_name: str
    ) -> bool:
        return intent_name in self.intents

    def get_intent(
        self,
        intent_name: str
    ) -> Optional[dict]:
        return self.intents.get(intent_name)

    def all_intent_names(self) -> List[str]:
        return sorted(self.intents.keys())

    def validate_intent_definition(
        self,
        intent_name: str
    ) -> bool:

        intent = self.get_intent(intent_name)

        if not intent:
            return False

        domain = intent.get("domain")
        category = intent.get("category")

        return (
            domain in DOMAINS
            and category in CATEGORIES
            and category_matches_domain(domain, category)
        )

    def validate_registry(self) -> List[str]:
        """
        Return a list of invalid registry definitions.
        """

        errors = []

        for intent_name in self.intents:

            if not self.validate_intent_definition(intent_name):
                errors.append(intent_name)

        return errors


# ---------------------------------------------------------
# Initialize Registry
# ---------------------------------------------------------
registry_path = os.path.join(
    config.config_dir,
    "intent_registry.json"
)

registry = IntentRegistry(
    registry_path,
    default_registry=DEFAULT_INTENT_REGISTRY
)

registry_errors = registry.validate_registry()

if registry_errors:
    raise ValueError(
        "Invalid intent definitions detected: "
        + ", ".join(registry_errors)
    )

print(
    f"Loaded IntentRegistry: "
    f"{len(registry.intents)} intents."
)

print(
    "Intent registry validation: PASSED"
)

## 7. Dataset Adapters
Adapters extract ONLY the original source text (`raw_text`) and the source name (`source`).
Real-world noise, mistakes, and speech artifacts are fully preserved.

In [ ]:
# 7. Dataset Adapters
# Extract ONLY original raw_text + source

def is_valid_raw_text(
    text: Optional[str]
) -> bool:
    """
    Validate whether extracted source text is non-empty.

    This function does NOT modify the source text.
    """

    if text is None:
        return False

    if not isinstance(text, str):
        text = str(text)

    return len(text.strip()) > 0


class DatasetAdapter(ABC):

    def __init__(
        self,
        source_name: str,
        target_count: int,
        config: PipelineConfig
    ):
        self.source_name = source_name
        self.target_count = target_count
        self.config = config

    @abstractmethod
    def fetch_records(self) -> List[dict]:
        """
        Return records containing exactly:

            {
                "raw_text": <original source text>,
                "source": <source identifier>
            }

        Source labels and other metadata may be used internally,
        but must not be returned here.
        """
        pass

    def clean_record_text(
        self,
        text: Any
    ) -> Optional[str]:
        """
        Minimal validation-only operation.

        IMPORTANT:
        We only strip surrounding whitespace.
        We do not translate, paraphrase, correct,
        normalize, or rewrite the content.
        """

        if text is None:
            return None

        if not isinstance(text, str):
            text = str(text)

        text = text.strip()

        if not text:
            return None

        return text

    def build_record(
        self,
        raw_text: Any,
        source: Optional[str] = None
    ) -> Optional[dict]:

        text = self.clean_record_text(raw_text)

        if text is None:
            return None

        return {
            "raw_text": text,
            "source": source or self.source_name
        }

    def sample_hf_dataset(
        self,
        dataset,
        count: int
    ):
        """
        Deterministically sample a Hugging Face dataset.
        """

        count = min(count, len(dataset))

        if count <= 0:
            return dataset.select([])

        return dataset.shuffle(
            seed=self.config.random_seed
        ).select(range(count))


# ---------------------------------------------------------
# BANKING77
# ---------------------------------------------------------
class Banking77Adapter(DatasetAdapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "BANKING77",
            config.target_banking77,
            config
        )

    def fetch_records(self) -> List[dict]:

        from datasets import load_dataset

        ds = load_dataset(
            "banking77",
            split="train"
        )

        sample = self.sample_hf_dataset(
            ds,
            self.target_count
        )

        records = []

        for row in sample:

            record = self.build_record(
                row.get("text"),
                "BANKING77"
            )

            if record:
                records.append(record)

        return records


# ---------------------------------------------------------
# CLINC150
# ---------------------------------------------------------
class Clinc150Adapter(DatasetAdapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "CLINC150",
            config.target_clinc150,
            config
        )

    def fetch_records(self) -> List[dict]:

        from datasets import load_dataset

        ds = load_dataset(
            "DeepPavlov/clinc150",
            name="default",
            split="train"
        )

        sample = self.sample_hf_dataset(
            ds,
            self.target_count
        )

        records = []

        for row in sample:

            record = self.build_record(
                row.get("utterance"),
                "CLINC150"
            )

            if record:
                records.append(record)

        return records


# ---------------------------------------------------------
# HWU64
# ---------------------------------------------------------
class Hwu64Adapter(DatasetAdapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "HWU64",
            config.target_hwu64,
            config
        )

    def fetch_records(self) -> List[dict]:

        from datasets import load_dataset

        ds = load_dataset(
            "DeepPavlov/hwu64",
            name="default",
            split="train"
        )

        sample = self.sample_hf_dataset(
            ds,
            self.target_count
        )

        records = []

        for row in sample:

            record = self.build_record(
                row.get("utterance"),
                "HWU64"
            )

            if record:
                records.append(record)

        return records


# ---------------------------------------------------------
# MINDS-14
# ---------------------------------------------------------
class Minds14Adapter(DatasetAdapter):

    def _load_language(
        self,
        language: str,
        target: int,
        source_name: str
    ) -> List[dict]:

        from datasets import load_dataset

        ds = load_dataset(
            "PolyAI/minds14",
            name=language,
            split="train"
        )

        sample = self.sample_hf_dataset(
            ds,
            target
        )

        records = []

        for row in sample:

            # Prefer the English transcription field.
            text = (
                row.get("english_transcription")
                or row.get("transcription")
            )

            record = self.build_record(
                text,
                source_name
            )

            if record:
                records.append(record)

        return records


class Minds14USAdapter(Minds14Adapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "MINDS14_EN_US",
            config.target_minds14_us,
            config
        )

    def fetch_records(self) -> List[dict]:

        return self._load_language(
            "en-US",
            self.target_count,
            "MINDS14_EN_US"
        )


class Minds14AUAdapter(Minds14Adapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "MINDS14_EN_AU",
            config.target_minds14_au,
            config
        )

    def fetch_records(self) -> List[dict]:

        return self._load_language(
            "en-AU",
            self.target_count,
            "MINDS14_EN_AU"
        )


class Minds14GBAdapter(Minds14Adapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "MINDS14_EN_GB",
            config.target_minds14_gb,
            config
        )

    def fetch_records(self) -> List[dict]:

        return self._load_language(
            "en-GB",
            self.target_count,
            "MINDS14_EN_GB"
        )


# ---------------------------------------------------------
# CORD-v2
# ---------------------------------------------------------
class CordV2Adapter(DatasetAdapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "CORD-v2",
            config.target_cord_v2,
            config
        )

    def fetch_records(self) -> List[dict]:

        from datasets import load_dataset

        ds = load_dataset(
            "naver-clova-ix/cord-v2",
            split="train"
        )

        sample = self.sample_hf_dataset(
            ds,
            self.target_count
        )

        records = []

        for row in sample:

            # CORD-v2 contains document/receipt images and
            # structured annotations. We must not reconstruct
            # synthetic text from ground-truth annotations.
            #
            # This adapter therefore requires an actual text
            # field from the dataset representation.

            text = None

            for field_name in [
                "text",
                "document_text",
                "ocr_text"
            ]:
                if field_name in row:
                    text = row.get(field_name)
                    if is_valid_raw_text(text):
                        break

            record = self.build_record(
                text,
                "CORD-v2"
            )

            if record:
                records.append(record)

        return records


# ---------------------------------------------------------
# SROIE
# ---------------------------------------------------------
class SroieAdapter(DatasetAdapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "SROIE",
            config.target_sroie,
            config
        )

    def fetch_records(self) -> List[dict]:

        from datasets import load_dataset

        ds = load_dataset(
            "rth/sroie-2019-v2",
            split="train"
        )

        sample = self.sample_hf_dataset(
            ds,
            self.target_count
        )

        records = []

        for row in sample:

            text = None

            for field_name in [
                "text",
                "document_text",
                "ocr_text"
            ]:
                if field_name in row:
                    text = row.get(field_name)
                    if is_valid_raw_text(text):
                        break

            record = self.build_record(
                text,
                "SROIE"
            )

            if record:
                records.append(record)

        return records


# ---------------------------------------------------------
# FUNSD
# ---------------------------------------------------------
class FunsdAdapter(DatasetAdapter):

    def __init__(self, config: PipelineConfig):
        super().__init__(
            "FUNSD",
            config.target_funsd,
            config
        )

    def fetch_records(self) -> List[dict]:

        from datasets import load_dataset

        ds = load_dataset(
            "nielsr/funsd",
            split="train"
        )

        sample = self.sample_hf_dataset(
            ds,
            self.target_count
        )

        records = []

        for row in sample:

            # FUNSD provides OCR word annotations.
            # Joining those original words reconstructs the
            # source OCR text without inventing semantic content.
            words = row.get("words")

            if isinstance(words, list):
                text = " ".join(
                    str(word)
                    for word in words
                )
            else:
                text = None

            record = self.build_record(
                text,
                "FUNSD"
            )

            if record:
                records.append(record)

        return records


print(
    "All dataset adapters defined."
)

print(
    "Adapters return only: raw_text + source."
)

print(
    "No semantic annotation or synthetic scenario generation "
    "occurs during source extraction."
)

## 8. Original Dataset Builder
Builds the unified dataset containing ONLY `raw_text` and `source` (exactly 23,384 rows, 0 shortfall).

In [ ]:
# 8. Original Dataset Builder Execution

adapters = [
    Banking77Adapter(config),
    Clinc150Adapter(config),
    Hwu64Adapter(config),

    Minds14USAdapter(config),
    Minds14AUAdapter(config),
    Minds14GBAdapter(config),

    CordV2Adapter(config),
    SroieAdapter(config),
    FunsdAdapter(config),
]


all_original_records = []
adapter_stats = []


for adapter in adapters:

    start_time = time.time()

    try:
        records = adapter.fetch_records()

    except Exception as exc:

        print(
            f"[{adapter.source_name:16}] "
            f"FAILED: {type(exc).__name__}: {exc}"
        )

        adapter_stats.append({
            "source": adapter.source_name,
            "requested": adapter.target_count,
            "available": 0,
            "valid": 0,
            "excluded_empty": 0,
            "selected": 0,
            "shortfall": adapter.target_count,
            "status": "FAILED",
            "error": str(exc),
        })

        continue


    available = len(records)

    valid_records = []
    excluded_count = 0

    for record in records:

        raw_text = record.get("raw_text")

        if is_valid_raw_text(raw_text):

            valid_records.append({
                "raw_text": str(raw_text).strip(),
                "source": record["source"],
            })

        else:
            excluded_count += 1


    selected = valid_records[:adapter.target_count]

    shortfall = max(
        0,
        adapter.target_count - len(selected)
    )


    all_original_records.extend(
        selected
    )


    elapsed = time.time() - start_time

    stats = {
        "source": adapter.source_name,
        "requested": adapter.target_count,
        "available": available,
        "valid": len(valid_records),
        "excluded_empty": excluded_count,
        "selected": len(selected),
        "shortfall": shortfall,
        "status": "OK",
        "error": "",
    }

    adapter_stats.append(stats)

    print(
        f"[{adapter.source_name:16}] "
        f"Avail: {available:6,} | "
        f"Valid: {len(valid_records):6,} | "
        f"Selected: {len(selected):6,} | "
        f"Shortfall: {shortfall:6,} | "
        f"{elapsed:.2f}s"
    )


# ---------------------------------------------------------
# Build Original Dataset
# ---------------------------------------------------------
df_original = pd.DataFrame(
    all_original_records,
    columns=["raw_text", "source"]
)


print()
print(
    f"Total Unified Original Records: "
    f"{len(df_original):,}"
)

print(
    f"Requested Baseline: "
    f"{config.total_target:,}"
)

print(
    f"Overall Shortfall: "
    f"{max(0, config.total_target - len(df_original)):,}"
)

## 9. Source Data Inspection
Verifies that the original dataset contains exactly two columns: `raw_text` and `source`.

In [ ]:
# 9. Verify Schema and Inspect Samples Per Source

# ---------------------------------------------------------
# Strict Original Dataset Schema
# ---------------------------------------------------------
expected_columns = [
    config.original_raw_text_column,
    config.original_source_column,
]

assert list(df_original.columns) == expected_columns, (
    f"Columns violation: {list(df_original.columns)}"
)

assert df_original["raw_text"].notna().all(), (
    "Null raw_text found!"
)

assert df_original["source"].notna().all(), (
    "Null source found!"
)

assert (
    df_original["raw_text"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "Empty raw_text found!"
)


# ---------------------------------------------------------
# Check for Exact Duplicates
#
# Only exact duplicates of raw_text + source are considered
# duplicates here. We do NOT aggressively remove semantically
# similar utterances.
# ---------------------------------------------------------
duplicate_mask = df_original.duplicated(
    subset=["raw_text", "source"],
    keep=False
)

duplicate_count = int(
    duplicate_mask.sum()
)

print(
    "Original dataset schema verified:"
)

print(
    f"  Columns: {list(df_original.columns)}"
)

print(
    f"  Records: {len(df_original):,}"
)

print(
    f"  Requested target: {config.total_target:,}"
)

print(
    f"  Exact duplicate rows: {duplicate_count:,}"
)


# ---------------------------------------------------------
# Source Distribution
# ---------------------------------------------------------
print()
print("--- RECORD COUNT PER SOURCE ---")

source_counts = (
    df_original["source"]
    .value_counts()
    .sort_index()
)

print(
    source_counts.to_string()
)


# ---------------------------------------------------------
# Sample Inspection
# ---------------------------------------------------------
print()
print("--- 3 SAMPLES PER SOURCE ---")

for source_name in sorted(
    df_original["source"].unique()
):

    print()
    print(f"[Source: {source_name}]")

    subset = df_original[
        df_original["source"] == source_name
    ]

    sample_count = min(
        3,
        len(subset)
    )

    if sample_count == 0:
        continue

    samples = subset.sample(
        n=sample_count,
        random_state=config.random_seed
    )

    for index, (_, row) in enumerate(
        samples.iterrows(),
        start=1
    ):

        preview = (
            str(row["raw_text"])
            .replace("\n", " ")
            .replace("\r", " ")
        )

        if len(preview) > 120:
            preview = preview[:120] + "..."

        print(
            f"  {index}. {preview}"
        )

## 10. Sampling & Shortfall Audit
Audit table comparing requested targets vs selected records.

In [ ]:
# 10. Original Dataset Audit

audit_df = pd.DataFrame(
    adapter_stats
)


# ---------------------------------------------------------
# Per-Source Audit
# ---------------------------------------------------------
audit_columns = [
    "source",
    "requested",
    "available",
    "valid",
    "excluded_empty",
    "selected",
    "shortfall",
    "status",
]

print(
    audit_df[
        audit_columns
    ].to_string(index=False)
)


# ---------------------------------------------------------
# Aggregate Audit
# ---------------------------------------------------------
total_requested = int(
    audit_df["requested"].sum()
)

total_available = int(
    audit_df["available"].sum()
)

total_valid = int(
    audit_df["valid"].sum()
)

total_selected = int(
    audit_df["selected"].sum()
)

total_shortfall = int(
    audit_df["shortfall"].sum()
)

total_excluded = int(
    audit_df["excluded_empty"].sum()
)


print()
print("========================================")
print("ORIGINAL DATASET AUDIT SUMMARY")
print("========================================")

print(
    f"Requested:       {total_requested:,}"
)

print(
    f"Available:       {total_available:,}"
)

print(
    f"Valid:           {total_valid:,}"
)

print(
    f"Selected:        {total_selected:,}"
)

print(
    f"Excluded empty:  {total_excluded:,}"
)

print(
    f"Shortfall:       {total_shortfall:,}"
)

print(
    f"Configured total target: {config.total_target:,}"
)

print(
    f"Actual dataset size:     {len(df_original):,}"
)

print("========================================")


# ---------------------------------------------------------
# Important: Do NOT fail when a source is short.
# ---------------------------------------------------------
if total_shortfall > 0:

    print()
    print(
        "WARNING: The requested baseline target was not fully "
        "reached."
    )

    print(
        "No records will be duplicated or fabricated to "
        "fill the shortfall."
    )

else:

    print()
    print(
        "All requested source targets were reached."
    )

## 11. Original Dataset Export
Save `original_dataset.csv` with strictly `raw_text` and `source`.

In [ ]:
# 11. Save original_dataset.csv
# The original dataset is immutable source content.
# It must contain exactly two columns: raw_text and source.

os.makedirs(config.output_dir, exist_ok=True)

orig_csv_path = os.path.join(
    config.output_dir,
    "original_dataset.csv"
)

required_columns = ["raw_text", "source"]

if list(df_original.columns) != required_columns:
    raise ValueError(
        f"original_dataset.csv must contain exactly {required_columns}. "
        f"Found: {list(df_original.columns)}"
    )

if df_original.empty:
    raise ValueError(
        "original_dataset.csv cannot be empty. "
        "No source records were successfully extracted."
    )

df_original.to_csv(
    orig_csv_path,
    index=False,
    encoding="utf-8"
)

print(
    f"Saved {orig_csv_path} "
    f"({len(df_original):,} rows, "
    f"{os.path.getsize(orig_csv_path):,} bytes)"
)

print("Columns:", list(df_original.columns))
print("Original source layer locked: raw_text + source only.")

## 12. LLM Provider Abstraction
OpenRouter client with strict quota and retry controls.

In [ ]:
# 12. OpenRouter Provider

class LLMProvider(ABC):
    @abstractmethod
    def generate(
        self,
        prompt: str,
        system_prompt: Optional[str] = None,
        is_test: bool = False,
        stage: str = "annotation"
    ) -> str:
        pass


class OpenRouterProvider(LLMProvider):
    def __init__(
        self,
        config: PipelineConfig,
        ledger: RequestLedger
    ):
        self.config = config
        self.ledger = ledger

        self.api_key = os.environ.get(
            "OPENROUTER_API_KEY",
            ""
        )

        try:
            from google.colab import userdata

            self.api_key = (
                self.api_key
                or userdata.get("OPENROUTER_API_KEY", "")
            )
        except Exception:
            pass

    def is_configured(self) -> bool:
        return bool(self.api_key)

    def _get_model(self, stage: str) -> str:
        model_map = {
            "annotation": self.config.annotation_model,
            "synthetic": self.config.synthetic_model,
            "validation": self.config.validation_model,
            "connectivity_test": self.config.connectivity_test_model,
        }

        if stage not in model_map:
            raise ValueError(
                f"Unknown LLM stage: {stage}"
            )

        model = model_map[stage]

        if not model:
            raise ValueError(
                f"No model configured for stage '{stage}'."
            )

        return model

    def generate(
        self,
        prompt: str,
        system_prompt: Optional[str] = None,
        is_test: bool = False,
        stage: str = "annotation"
    ) -> str:

        if not self.is_configured():
            raise RuntimeError(
                "OPENROUTER_API_KEY is not configured."
            )

        model = self._get_model(stage)

        if (
            not is_test
            and not self.ledger.can_request()
        ):
            raise RuntimeError(
                "Production request limit reached "
                f"({self.config.daily_request_limit}/day)."
            )

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://latentspace.ai",
            "X-Title": "LatentSpace Dataset Pipeline"
        }

        messages = []

        if system_prompt:
            messages.append({
                "role": "system",
                "content": system_prompt
            })

        messages.append({
            "role": "user",
            "content": prompt
        })

        payload = {
            "model": model,
            "messages": messages,
            "response_format": {
                "type": "json_object"
            }
        }

        request_url = (
            f"{self.config.api_base_url}"
            "/chat/completions"
        )

        req = urllib.request.Request(
            request_url,
            data=json.dumps(
                payload,
                ensure_ascii=False
            ).encode("utf-8"),
            headers=headers,
            method="POST"
        )

        t0 = time.time()

        attempt_id = (
            f"req_{int(time.time() * 1000)}"
        )

        try:
            with urllib.request.urlopen(
                req,
                timeout=self.config.request_timeout
            ) as resp:

                response_bytes = resp.read()

                body = json.loads(
                    response_bytes.decode("utf-8")
                )

                latency = int(
                    (time.time() - t0) * 1000
                )

                choices = body.get("choices", [])

                if not choices:
                    raise RuntimeError(
                        "OpenRouter response contains no choices."
                    )

                message = choices[0].get(
                    "message",
                    {}
                )

                content = message.get(
                    "content"
                )

                if not isinstance(
                    content,
                    str
                ) or not content.strip():

                    raise RuntimeError(
                        "OpenRouter returned an empty "
                        "or non-text response."
                    )

                self.ledger.record_attempt({
                    "timestamp": datetime.now(
                        timezone.utc
                    ).isoformat(),

                    "attempt_id": attempt_id,
                    "is_test": is_test,
                    "model": model,
                    "http_status": resp.status,
                    "error_category": None,
                    "latency_ms": latency
                })

                return content

        except Exception as e:

            latency = int(
                (time.time() - t0) * 1000
            )

            status_code = getattr(
                e,
                "code",
                500
            )

            self.ledger.record_attempt({
                "timestamp": datetime.now(
                    timezone.utc
                ).isoformat(),

                "attempt_id": attempt_id,
                "is_test": is_test,
                "model": model,
                "http_status": status_code,
                "error_category": type(e).__name__,
                "latency_ms": latency
            })

            raise


provider = OpenRouterProvider(
    config,
    ledger
)

print(
    "Provider initialized."
)

print(
    f"Configured: {provider.is_configured()}"
)

print(
    f"Annotation model: "
    f"{config.annotation_model}"
)

## 13. Pre-Flight Connectivity Test
Validates OpenRouter API access without consuming production quota.

In [ ]:
# 13. Pre-Flight Test

if provider.is_configured():

    try:
        test_prompt = (
            'Respond with ONLY this JSON object: '
            '{"status":"ok"}'
        )

        resp = provider.generate(
            prompt=test_prompt,
            system_prompt=(
                "Return valid JSON only."
            ),
            is_test=True,
            stage="connectivity_test"
        )

        parsed_test = json.loads(
            resp.strip()
        )

        if parsed_test.get("status") != "ok":
            raise ValueError(
                f"Unexpected test response: {parsed_test}"
            )

        print(
            "Pre-flight connectivity test succeeded."
        )

        print(
            f"Response: {json.dumps(parsed_test)}"
        )

    except Exception as e:

        print(
            "Pre-flight test failed:"
        )

        print(
            f"{type(e).__name__}: {e}"
        )

else:

    print(
        "Pre-flight test skipped: "
        "OPENROUTER_API_KEY is not configured."
    )

    print(
        "Strict True LLM mode remains active."
    )

## 14. Agentic Schema Completion & Synthesis Engine
Transforms raw_text into structured agentic actions using true LLM provider.
Strict True LLM Only: Never uses fake heuristic fallback. Failed annotations are isolated.

In [ ]:
# 14. Annotation Engine (Strict True LLM)

class AnnotationEngine:

    SYSTEM_PROMPT = """
You are the semantic middleware annotation engine for LatentSpace.

Your job is to analyze ONE user raw-text record and convert it into
a structured action plan.

IMPORTANT SOURCE RULES:
1. Treat the raw text as immutable source content.
2. Do not rewrite, correct, translate, paraphrase, or normalize the raw text.
3. Do not invent facts that are not supported by the raw text.
4. The source dataset name is provenance only. Do not infer intent from
   the dataset name or source label.
5. Do not fabricate tool execution results.

INTENT RULES:
1. You may ONLY use intent_name values that exist in the supplied
   Intent Registry.
2. Never invent a new intent name.
3. If the user intent is unclear, use:
   clarification_required
4. If the request is outside the supported assistant scope, use:
   out_of_scope
5. Do not use a domain as a substitute for an intent.
6. Do not assume a default intent merely because a domain is obvious.
7. Use multiple actions only when the raw text genuinely contains
   multiple distinct user intents.
8. Use depends_on only when one action logically depends on another.
9. Independent actions must not depend on each other.
10. Do not manufacture multi-step workflows.

ENTITY RULES:
1. Extract only entities supported by the raw text.
2. Missing required information must be placed in missing_information.
3. Never invent names, dates, amounts, account numbers, locations,
   recipients, booking details, or other values.
4. Preserve user-provided values faithfully.

CONFIRMATION RULES:
Set requires_confirmation=true when the requested action would normally
require explicit confirmation before execution or when important
information is still unresolved.

TOOL RULES:
If one or more valid actions exist, emit exactly one dispatch_actions
tool call containing those actions.

The tool call must have this structure:

{
  "id": "call_001",
  "type": "function",
  "function": {
    "name": "dispatch_actions",
    "arguments": "{\"actions\":[...]}"
  }
}

The model is NOT executing the tool.
The tool call is only a structured request for the downstream system.

If there are no executable actions:
- intents must be []
- tool_calls must be []
- expected_response must contain either a useful clarification
  question or an appropriate out-of-scope response.

OUTPUT RULES:
Return exactly ONE valid JSON object.
Do not use Markdown.
Do not use ```json fences.

Required top-level keys:

{
  "domain": "...",
  "category": "...",
  "goal": "...",
  "intents": [],
  "entities": {},
  "missing_information": [],
  "requires_confirmation": false,
  "execution_plan": [],
  "tool_calls": [],
  "expected_response": "..."
}

DOMAIN TAXONOMY:
- finance
- productivity
- travel
- commerce
- communication
- account_and_security
- documents
- information
- other

CATEGORY TAXONOMY:
- finance/account
- finance/transactions
- finance/payments
- finance/transfers
- finance/cards
- finance/expenses
- productivity/tasks
- productivity/reminders
- productivity/alarms
- productivity/notes
- productivity/calendar
- travel/flights
- travel/reservations
- travel/transportation
- commerce/products
- commerce/orders
- commerce/subscriptions
- commerce/purchases
- communication/messages
- communication/email
- communication/contacts
- account_and_security/profile
- account_and_security/authentication
- account_and_security/credentials
- account_and_security/security
- account_and_security/cards
- documents/ocr
- documents/extraction
- documents/receipts
- documents/invoices
- documents/forms
- information/general_information
- information/navigation
- information/calculation
- other/clarification
- other/out_of_scope
"""

    def __init__(
        self,
        config: PipelineConfig,
        registry: IntentRegistry,
        provider: Optional[LLMProvider] = None
    ):
        self.config = config
        self.registry = registry
        self.provider = provider

        self.cache_dir = os.path.join(
            config.cache_dir,
            "annotations"
        )

        os.makedirs(
            self.cache_dir,
            exist_ok=True
        )

    @staticmethod
    def _strip_json_fences(text: str) -> str:
        text = text.strip()

        if text.startswith("```json"):
            text = text[7:]

        elif text.startswith("```"):
            text = text[3:]

        if text.endswith("```"):
            text = text[:-3]

        return text.strip()

    def annotate_record(
        self,
        record_id: str,
        raw_text: str,
        source: str
    ) -> Tuple[Optional[dict], Optional[dict]]:

        cache_path = os.path.join(
            self.cache_dir,
            f"{record_id}.json"
        )

        if os.path.exists(cache_path):

            cached = CheckpointManager.load_json(
                cache_path
            )

            if (
                cached
                and cached.get(
                    "validation_status"
                ) == "accepted"
            ):
                return cached, None

        if (
            self.provider is None
            or not self.provider.is_configured()
        ):
            return None, None

        user_content = (
            "INTENT REGISTRY:\n"
            + json.dumps(
                getattr(
                    self.registry,
                    "registry",
                    {}
                ),
                ensure_ascii=False,
                indent=2
            )
            + "\n\n"
            "SOURCE DATASET: "
            + str(source)
            + "\n\n"
            "RAW TEXT:\n"
            + str(raw_text)
        )

        try:

            resp_str = self.provider.generate(
                prompt=user_content,
                system_prompt=self.SYSTEM_PROMPT,
                is_test=False,
                stage="annotation"
            )

            clean_str = self._strip_json_fences(
                resp_str
            )

            parsed = json.loads(
                clean_str
            )

            if not isinstance(parsed, dict):
                raise ValueError(
                    "LLM response must be a JSON object."
                )

            annotated = {
                "record_id": record_id,
                "source": source,
                "raw_text": raw_text,

                "domain": parsed.get(
                    "domain"
                ),

                "category": parsed.get(
                    "category"
                ),

                "goal": parsed.get(
                    "goal",
                    ""
                ),

                "intents": parsed.get(
                    "intents",
                    []
                ),

                "entities": parsed.get(
                    "entities",
                    {}
                ),

                "missing_information": parsed.get(
                    "missing_information",
                    []
                ),

                "requires_confirmation": parsed.get(
                    "requires_confirmation",
                    False
                ),

                "execution_plan": parsed.get(
                    "execution_plan",
                    []
                ),

                "tool_calls": parsed.get(
                    "tool_calls",
                    []
                ),

                "expected_response": parsed.get(
                    "expected_response",
                    ""
                ),

                "model": self.config.annotation_model,

                "validation_status": "pending"
            }

            return annotated, None

        except Exception as e:

            failed_record = {
                "record_id": record_id,
                "source": source,
                "raw_text": raw_text,
                "error_category": (
                    "llm_call_or_parse_error"
                ),
                "error_message": str(e),
                "raw_llm_response": (
                    resp_str
                    if "resp_str" in locals()
                    else ""
                ),
                "timestamp": datetime.now(
                    timezone.utc
                ).isoformat()
            }

            return None, failed_record


annotation_engine = AnnotationEngine(
    config,
    registry,
    provider
)

print(
    "AnnotationEngine initialized "
    "in Strict True LLM mode."
)

## 15. Validation Engine
Deterministic structural validation against IntentRegistry and `dispatch_actions` universal tool contract.

In [ ]:
# 15. Validation Engine

class ValidationEngine:

    def __init__(
        self,
        registry: IntentRegistry
    ):
        self.registry = registry

    def _get_intent_definition(
        self,
        intent_name: str
    ) -> Optional[dict]:

        # Support the registry implementation from Cell 6
        # without requiring a specific internal attribute name.

        registry_data = getattr(
            self.registry,
            "registry",
            None
        )

        if isinstance(registry_data, dict):
            definition = registry_data.get(
                intent_name
            )

            if isinstance(definition, dict):
                return definition

        registry_data = getattr(
            self.registry,
            "_registry",
            None
        )

        if isinstance(registry_data, dict):
            definition = registry_data.get(
                intent_name
            )

            if isinstance(definition, dict):
                return definition

        return None

    def validate(
        self,
        annotated: dict
    ) -> Tuple[bool, Optional[str], Optional[str]]:

        required_fields = [
            "domain",
            "category",
            "intents",
            "entities",
            "missing_information",
            "requires_confirmation",
            "execution_plan",
            "tool_calls",
            "expected_response"
        ]

        for field in required_fields:

            if field not in annotated:
                return (
                    False,
                    "missing_output_field",
                    f"Missing required field '{field}'."
                )

        domain = annotated.get("domain")
        category = annotated.get("category")
        intents = annotated.get("intents")
        entities = annotated.get("entities")
        missing_information = annotated.get(
            "missing_information"
        )
        tool_calls = annotated.get(
            "tool_calls"
        )

        if domain not in DOMAINS:
            return (
                False,
                "invalid_domain",
                f"Domain '{domain}' not in taxonomy."
            )

        if category not in CATEGORIES:
            return (
                False,
                "invalid_category",
                f"Category '{category}' not in taxonomy."
            )

        if not category_matches_domain(
            domain,
            category
        ):
            return (
                False,
                "domain_category_mismatch",
                f"Category '{category}' does not belong "
                f"to domain '{domain}'."
            )

        if not isinstance(intents, list):
            return (
                False,
                "invalid_intents_type",
                "'intents' must be a list."
            )

        if not isinstance(entities, dict):
            return (
                False,
                "invalid_entities_type",
                "'entities' must be an object."
            )

        if not isinstance(
            missing_information,
            list
        ):
            return (
                False,
                "invalid_missing_information_type",
                "'missing_information' must be a list."
            )

        if not isinstance(tool_calls, list):
            return (
                False,
                "invalid_tool_calls_type",
                "'tool_calls' must be a list."
            )

        action_ids = set()

        for index, act in enumerate(intents):

            if not isinstance(act, dict):
                return (
                    False,
                    "invalid_action_object",
                    f"Intent at index {index} must be an object."
                )

            aid = act.get("action_id")
            iname = act.get("intent_name")
            depends_on = act.get(
                "depends_on",
                []
            )
            action_entities = act.get(
                "entities",
                {}
            )

            if not isinstance(
                aid,
                str
            ) or not re.fullmatch(
                r"^a[0-9]+$",
                aid
            ):
                return (
                    False,
                    "invalid_action_id",
                    f"Invalid action_id '{aid}'."
                )

            if aid in action_ids:
                return (
                    False,
                    "duplicate_action_id",
                    f"Duplicate action_id '{aid}'."
                )

            if not isinstance(
                iname,
                str
            ) or not iname.strip():
                return (
                    False,
                    "missing_intent_name",
                    f"Missing intent_name for '{aid}'."
                )

            if not self.registry.is_valid_intent(
                iname
            ):
                return (
                    False,
                    "unregistered_intent",
                    f"Intent '{iname}' is not registered."
                )

            definition = self._get_intent_definition(
                iname
            )

            if definition:

                intent_domain = definition.get(
                    "domain"
                )

                intent_category = definition.get(
                    "category"
                )

                if (
                    intent_domain
                    and intent_domain != domain
                ):
                    return (
                        False,
                        "intent_domain_mismatch",
                        f"Intent '{iname}' belongs to "
                        f"domain '{intent_domain}', not '{domain}'."
                    )

                if (
                    intent_category
                    and intent_category != category
                ):
                    return (
                        False,
                        "intent_category_mismatch",
                        f"Intent '{iname}' belongs to "
                        f"category '{intent_category}', "
                        f"not '{category}'."
                    )

            if not isinstance(
                depends_on,
                list
            ):
                return (
                    False,
                    "invalid_dependency_type",
                    f"depends_on for '{aid}' must be a list."
                )

            for dep in depends_on:

                if dep not in action_ids:
                    return (
                        False,
                        "invalid_dependency",
                        f"Dependency '{dep}' must reference "
                        f"an earlier action."
                    )

            if not isinstance(
                action_entities,
                dict
            ):
                return (
                    False,
                    "invalid_action_entities",
                    f"entities for '{aid}' must be an object."
                )

            action_ids.add(aid)

        # No actions means no tool call.
        if not intents:

            if tool_calls:
                return (
                    False,
                    "unexpected_tool_call",
                    "tool_calls must be empty when intents is empty."
                )

        # Actions require exactly one dispatch_actions call.
        else:

            if len(tool_calls) != 1:
                return (
                    False,
                    "invalid_tool_call_count",
                    "Exactly one dispatch_actions "
                    "tool call is required when actions exist."
                )

        for tc in tool_calls:

            if not isinstance(tc, dict):
                return (
                    False,
                    "invalid_tool_call_object",
                    "Each tool call must be an object."
                )

            if tc.get("type") != "function":
                return (
                    False,
                    "invalid_tool_call_type",
                    "Tool call type must be 'function'."
                )

            function = tc.get(
                "function",
                {}
            )

            if not isinstance(
                function,
                dict
            ):
                return (
                    False,
                    "invalid_function_object",
                    "Tool call function must be an object."
                )

            if function.get("name") != "dispatch_actions":
                return (
                    False,
                    "invalid_tool_name",
                    "Tool name must be 'dispatch_actions'."
                )

            arguments = function.get(
                "arguments"
            )

            if not isinstance(
                arguments,
                str
            ):
                return (
                    False,
                    "invalid_tool_arguments_type",
                    "Tool arguments must be a JSON string."
                )

            try:
                args = json.loads(
                    arguments
                )
            except Exception as e:
                return (
                    False,
                    "malformed_json_arguments",
                    f"Tool arguments JSON parse error: {e}"
                )

            if not isinstance(args, dict):
                return (
                    False,
                    "invalid_tool_arguments_object",
                    "Tool arguments must decode to an object."
                )

            actions = args.get(
                "actions"
            )

            if not isinstance(
                actions,
                list
            ):
                return (
                    False,
                    "missing_actions_list",
                    "Tool arguments must contain an actions list."
                )

            if len(actions) != len(intents):
                return (
                    False,
                    "tool_action_count_mismatch",
                    "Tool-call actions must match intents exactly."
                )

            intent_ids = [
                x.get("action_id")
                for x in intents
            ]

            tool_ids = [
                x.get("action_id")
                for x in actions
            ]

            if intent_ids != tool_ids:
                return (
                    False,
                    "tool_action_id_mismatch",
                    "Tool-call action IDs must exactly match "
                    "intent action IDs and ordering."
                )

            for action in actions:

                if not isinstance(
                    action,
                    dict
                ):
                    return (
                        False,
                        "invalid_tool_action",
                        "Each tool action must be an object."
                    )

                allowed_keys = {
                    "action_id",
                    "intent_name",
                    "depends_on",
                    "entities"
                }

                if set(action.keys()) != allowed_keys:
                    return (
                        False,
                        "invalid_tool_action_keys",
                        "Tool action contains unexpected "
                        "or missing keys."
                    )

        return True, None, None


validation_engine = ValidationEngine(
    registry
)

print(
    "ValidationEngine initialized "
    "with strict taxonomy and tool validation."
)

## 16. Synthetic Dataset Extraction
Extracts only the verified LLM-annotated records. Unannotated or failed records are cleanly separated.

In [ ]:
# 16. Synthetic Dataset Extraction Execution

annotated_records = []
failed_records = []

if provider.is_configured():

    print(
        "Annotating records with True LLM..."
    )

    print(
        f"Model: {config.annotation_model}"
    )

    print(
        f"Daily production request limit: "
        f"{config.daily_request_limit}"
    )

    for idx, row in df_original.iterrows():

        # The ledger is the authoritative quota source.
        if not ledger.can_request():

            print(
                "Production request budget exhausted."
            )

            break

        record_id = (
            f"{row['source']}_{idx:06d}"
        )

        raw_text = str(
            row["raw_text"]
        )

        source = str(
            row["source"]
        )

        ann, fail = (
            annotation_engine.annotate_record(
                record_id=record_id,
                raw_text=raw_text,
                source=source
            )
        )

        if fail:

            failed_records.append(
                fail
            )

            continue

        if ann is None:
            continue

        is_valid, err_cat, err_msg = (
            validation_engine.validate(
                ann
            )
        )

        if is_valid:

            ann["validation_status"] = (
                "accepted"
            )

            cache_path = os.path.join(
                annotation_engine.cache_dir,
                f"{record_id}.json"
            )

            CheckpointManager.atomic_write_json(
                cache_path,
                ann
            )

            annotated_records.append(
                ann
            )

        else:

            failed_records.append({
                "record_id": record_id,
                "source": source,
                "raw_text": raw_text,
                "error_category": (
                    err_cat
                    or "validation_failure"
                ),
                "error_message": (
                    err_msg
                    or "Failed schema validation"
                ),
                "raw_llm_response": json.dumps(
                    ann,
                    ensure_ascii=False
                ),
                "timestamp": datetime.now(
                    timezone.utc
                ).isoformat()
            })

else:

    print(
        "Strict True LLM Mode: "
        "No API key configured."
    )

    print(
        "0 synthetic records generated."
    )

print()
print(
    f"Original records available: "
    f"{len(df_original):,}"
)

print(
    f"Accepted True LLM annotations: "
    f"{len(annotated_records):,}"
)

print(
    f"Failed annotations: "
    f"{len(failed_records):,}"
)

print(
    f"Production requests used: "
    f"{ledger.production_requests_used:,}"
)

print(
    f"Production requests remaining: "
    f"{ledger.remaining_requests():,}"
)

## 17. Deduplication & Integrity Check
Checks for semantic and lexical integrity across annotated records.

In [ ]:
# 17. Deduplication & Integrity Audit

rec_ids = [
    r["record_id"]
    for r in annotated_records
]

raw_text_keys = [
    (
        r.get("raw_text", "").strip().lower(),
        r.get("source", "")
    )
    for r in annotated_records
]

print(
    f"Total annotated records: "
    f"{len(annotated_records):,}"
)

print(
    f"Unique record IDs: "
    f"{len(set(rec_ids)):,}"
)

print(
    f"Unique source/text pairs: "
    f"{len(set(raw_text_keys)):,}"
)

if len(rec_ids) != len(set(rec_ids)):
    raise AssertionError(
        "Duplicate record IDs found!"
    )

if len(raw_text_keys) != len(set(raw_text_keys)):
    raise AssertionError(
        "Duplicate raw_text + source pairs "
        "found in annotated records!"
    )

for record in annotated_records:

    if not record.get("raw_text"):
        raise AssertionError(
            f"Empty raw_text for "
            f"{record.get('record_id')}"
        )

    if not record.get("source"):
        raise AssertionError(
            f"Missing source for "
            f"{record.get('record_id')}"
        )

    if record.get(
        "validation_status"
    ) != "accepted":

        raise AssertionError(
            f"Unaccepted record present in "
            f"annotated_records: "
            f"{record.get('record_id')}"
        )

print(
    "Integrity check passed."
)

## 18. Train / Validation / Test Split (Zero Leakage)
Executes group-aware 80% Train, 10% Validation, 10% Test split based on raw_text to ensure absolute zero text leakage.

In [ ]:
# 18. Zero-Leakage Split Manager

class SplitManager:

    @staticmethod
    def assign_splits(
        records: List[dict],
        seed: int = 42
    ) -> Tuple[
        List[dict],
        List[dict],
        List[dict]
    ]:

        if not records:
            return [], [], []

        rng = random.Random(
            seed
        )

        # Group exact normalized raw text together.
        # This prevents identical source text from crossing splits.
        groups = {}

        for record in records:

            key = (
                record.get(
                    "raw_text",
                    ""
                )
                .strip()
                .lower()
            )

            groups.setdefault(
                key,
                []
            ).append(record)

        unique_keys = list(
            groups.keys()
        )

        rng.shuffle(
            unique_keys
        )

        total_records = len(
            records
        )

        target_train = int(
            round(
                total_records * 0.80
            )
        )

        target_val = int(
            round(
                total_records * 0.10
            )
        )

        train = []
        val = []
        test = []

        for key in unique_keys:

            group = groups[key]

            remaining = (
                total_records
                - len(train)
                - len(val)
                - len(test)
            )

            if (
                len(train) < target_train
                and len(train) + len(group)
                <= target_train
            ):

                train.extend(group)

            elif (
                len(val) < target_val
                and len(val) + len(group)
                <= target_val
            ):

                val.extend(group)

            else:

                test.extend(group)

        for record in train:
            record["split"] = "train"

        for record in val:
            record["split"] = "validation"

        for record in test:
            record["split"] = "test"

        return train, val, test

    @staticmethod
    def verify_no_leakage(
        train: List[dict],
        val: List[dict],
        test: List[dict]
    ) -> bool:

        record_sets = {
            "train": {
                r["record_id"]
                for r in train
            },
            "validation": {
                r["record_id"]
                for r in val
            },
            "test": {
                r["record_id"]
                for r in test
            }
        }

        if (
            record_sets["train"]
            & record_sets["validation"]
        ):
            raise AssertionError(
                "Record leakage between "
                "train and validation."
            )

        if (
            record_sets["train"]
            & record_sets["test"]
        ):
            raise AssertionError(
                "Record leakage between "
                "train and test."
            )

        if (
            record_sets["validation"]
            & record_sets["test"]
        ):
            raise AssertionError(
                "Record leakage between "
                "validation and test."
            )

        text_sets = {
            split: {
                r.get(
                    "raw_text",
                    ""
                ).strip().lower()
                for r in records
            }
            for split, records in {
                "train": train,
                "validation": val,
                "test": test
            }.items()
        }

        if (
            text_sets["train"]
            & text_sets["validation"]
        ):
            raise AssertionError(
                "Text leakage between "
                "train and validation."
            )

        if (
            text_sets["train"]
            & text_sets["test"]
        ):
            raise AssertionError(
                "Text leakage between "
                "train and test."
            )

        if (
            text_sets["validation"]
            & text_sets["test"]
        ):
            raise AssertionError(
                "Text leakage between "
                "validation and test."
            )

        return True


train_records, val_records, test_records = (
    SplitManager.assign_splits(
        annotated_records,
        seed=config.random_seed
    )
)

SplitManager.verify_no_leakage(
    train_records,
    val_records,
    test_records
)

print(
    f"Splits generated:"
)

print(
    f"Train      = {len(train_records):,}"
)

print(
    f"Validation = {len(val_records):,}"
)

print(
    f"Test       = {len(test_records):,}"
)

print(
    "Zero record and normalized-text leakage "
    "verified across splits."
)

## 19. Final JSONL Export
Exports OpenAI-style conversation JSONL files with canonical `dispatch_actions` tool call, and exports failed annotations to a separate file.

In [ ]:
# 19. Final JSONL Export & Failed Annotation Isolation

class DatasetExporter:

    SYSTEM_MESSAGE = (
        "You are a helpful, autonomous AI assistant. "
        "You help users achieve their goals by utilizing "
        "the tools provided to you."
    )

    @staticmethod
    def format_jsonl_record(
        record: dict
    ) -> dict:

        raw_text = record.get(
            "raw_text",
            ""
        )

        tool_calls = record.get(
            "tool_calls",
            []
        )

        expected_response = record.get(
            "expected_response",
            ""
        )

        messages = [
            {
                "role": "system",
                "content": DatasetExporter.SYSTEM_MESSAGE
            },
            {
                "role": "user",
                "content": raw_text
            }
        ]

        if tool_calls:

            messages.append({
                "role": "assistant",
                "content": "",
                "tool_calls": tool_calls
            })

        else:

            messages.append({
                "role": "assistant",
                "content": expected_response
            })

        return {
            "messages": messages,
            "tools": [
                UNIVERSAL_DISPATCH_TOOL
            ]
        }

    @staticmethod
    def _write_jsonl(
        path: str,
        records: List[dict]
    ):

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            for record in records:

                formatted = (
                    DatasetExporter
                    .format_jsonl_record(record)
                )

                f.write(
                    json.dumps(
                        formatted,
                        ensure_ascii=False
                    )
                    + "\n"
                )

    @staticmethod
    def export_all(
        config: PipelineConfig,
        annotated_records: List[dict],
        failed_records: List[dict],
        train_records: List[dict],
        val_records: List[dict],
        test_records: List[dict]
    ):

        os.makedirs(
            config.output_dir,
            exist_ok=True
        )

        # --------------------------------------------------
        # 1. Failed annotations
        # --------------------------------------------------

        failed_columns = [
            "record_id",
            "source",
            "raw_text",
            "error_category",
            "error_message",
            "raw_llm_response",
            "timestamp"
        ]

        df_failed = (
            pd.DataFrame(
                failed_records,
                columns=failed_columns
            )
            if failed_records
            else pd.DataFrame(
                columns=failed_columns
            )
        )

        failed_csv_path = os.path.join(
            config.output_dir,
            "failed_annotations.csv"
        )

        df_failed.to_csv(
            failed_csv_path,
            index=False,
            encoding="utf-8"
        )

        failed_jsonl_path = os.path.join(
            config.output_dir,
            "failed_annotations.jsonl"
        )

        DatasetExporter._write_raw_jsonl(
            failed_jsonl_path,
            failed_records
        )

        print(
            f"Exported failed_annotations.csv "
            f"({len(df_failed):,} rows)"
        )

        print(
            f"Exported failed_annotations.jsonl "
            f"({len(df_failed):,} rows)"
        )

        # --------------------------------------------------
        # 2. Synthetic dataset
        # --------------------------------------------------

        synth_columns = [
            "record_id",
            "source",
            "raw_text",
            "domain",
            "category",
            "goal",
            "intents",
            "entities",
            "missing_information",
            "requires_confirmation",
            "execution_plan",
            "tool_calls",
            "expected_response",
            "model"
        ]

        synthetic_records = []

        for record in annotated_records:

            synthetic_records.append({
                "record_id": record["record_id"],
                "source": record["source"],
                "raw_text": record["raw_text"],
                "domain": record["domain"],
                "category": record["category"],
                "goal": record["goal"],

                "intents": json.dumps(
                    record["intents"],
                    ensure_ascii=False
                ),

                "entities": json.dumps(
                    record["entities"],
                    ensure_ascii=False
                ),

                "missing_information": json.dumps(
                    record["missing_information"],
                    ensure_ascii=False
                ),

                "requires_confirmation": (
                    record["requires_confirmation"]
                ),

                "execution_plan": json.dumps(
                    record["execution_plan"],
                    ensure_ascii=False
                ),

                "tool_calls": json.dumps(
                    record["tool_calls"],
                    ensure_ascii=False
                ),

                "expected_response": (
                    record["expected_response"]
                ),

                "model": record.get(
                    "model",
                    config.annotation_model
                )
            })

        df_synth = (
            pd.DataFrame(
                synthetic_records,
                columns=synth_columns
            )
            if synthetic_records
            else pd.DataFrame(
                columns=synth_columns
            )
        )

        synth_path = os.path.join(
            config.output_dir,
            "synthetic_dataset.csv"
        )

        df_synth.to_csv(
            synth_path,
            index=False,
            encoding="utf-8"
        )

        print(
            f"Exported synthetic_dataset.csv "
            f"({len(df_synth):,} rows)"
        )

        # --------------------------------------------------
        # 3. Combined dataset
        # --------------------------------------------------

        comb_columns = [
            "record_id",
            "source",
            "raw_text",
            "domain",
            "category",
            "goal",
            "intents",
            "entities",
            "missing_information",
            "requires_confirmation",
            "execution_plan",
            "tool_calls",
            "expected_response",
            "validation_status",
            "split"
        ]

        combined_records = []

        for record in annotated_records:

            combined_records.append({
                "record_id": record["record_id"],
                "source": record["source"],
                "raw_text": record["raw_text"],
                "domain": record["domain"],
                "category": record["category"],
                "goal": record["goal"],

                "intents": json.dumps(
                    record["intents"],
                    ensure_ascii=False
                ),

                "entities": json.dumps(
                    record["entities"],
                    ensure_ascii=False
                ),

                "missing_information": json.dumps(
                    record["missing_information"],
                    ensure_ascii=False
                ),

                "requires_confirmation": (
                    record["requires_confirmation"]
                ),

                "execution_plan": json.dumps(
                    record["execution_plan"],
                    ensure_ascii=False
                ),

                "tool_calls": json.dumps(
                    record["tool_calls"],
                    ensure_ascii=False
                ),

                "expected_response": (
                    record["expected_response"]
                ),

                "validation_status": (
                    record["validation_status"]
                ),

                "split": record.get(
                    "split",
                    ""
                )
            })

        df_combined = (
            pd.DataFrame(
                combined_records,
                columns=comb_columns
            )
            if combined_records
            else pd.DataFrame(
                columns=comb_columns
            )
        )

        combined_path = os.path.join(
            config.output_dir,
            "combined_dataset.csv"
        )

        df_combined.to_csv(
            combined_path,
            index=False,
            encoding="utf-8"
        )

        print(
            f"Exported combined_dataset.csv "
            f"({len(df_combined):,} rows)"
        )

        # --------------------------------------------------
        # 4. Training JSONL files
        # --------------------------------------------------

        jsonl_exports = [
            (
                "latentspace_complete_dataset.jsonl",
                annotated_records
            ),
            (
                "latentspace_train.jsonl",
                train_records
            ),
            (
                "latentspace_validation.jsonl",
                val_records
            ),
            (
                "latentspace_test.jsonl",
                test_records
            )
        ]

        for filename, records in jsonl_exports:

            output_path = os.path.join(
                config.output_dir,
                filename
            )

            DatasetExporter._write_jsonl(
                output_path,
                records
            )

            print(
                f"Exported {filename} "
                f"({len(records):,} records)"
            )

    @staticmethod
    def _write_raw_jsonl(
        path: str,
        records: List[dict]
    ):

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            for record in records:

                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    )
                    + "\n"
                )


DatasetExporter.export_all(
    config=config,
    annotated_records=annotated_records,
    failed_records=failed_records,
    train_records=train_records,
    val_records=val_records,
    test_records=test_records
)

## 20. Reporting
Generates `output/dataset_report.json` compiling all distribution metrics, audit stats, and LLM usage ledger.

In [ ]:
# 20. Quality Report Generation

class QualityReporter:

    @staticmethod
    def generate_report(
        config: PipelineConfig,
        adapter_stats: List[dict],
        total_original_count: int,
        annotated_records: List[dict],
        failed_records: List[dict],
        train_records: List[dict],
        val_records: List[dict],
        test_records: List[dict],
        ledger: RequestLedger
    ) -> dict:

        domain_counts = {}
        category_counts = {}
        intent_counts = {}
        source_counts = {}
        split_counts = {
            "train": len(train_records),
            "validation": len(val_records),
            "test": len(test_records)
        }

        for record in annotated_records:

            domain = record.get(
                "domain",
                "unknown"
            )

            category = record.get(
                "category",
                "unknown"
            )

            source = record.get(
                "source",
                "unknown"
            )

            domain_counts[domain] = (
                domain_counts.get(
                    domain,
                    0
                ) + 1
            )

            category_counts[category] = (
                category_counts.get(
                    category,
                    0
                ) + 1
            )

            source_counts[source] = (
                source_counts.get(
                    source,
                    0
                ) + 1
            )

            for action in record.get(
                "intents",
                []
            ):

                intent_name = action.get(
                    "intent_name",
                    "unknown"
                )

                intent_counts[intent_name] = (
                    intent_counts.get(
                        intent_name,
                        0
                    ) + 1
                )

        annotated_ids = {
            record["record_id"]
            for record in annotated_records
        }

        failed_ids = {
            record["record_id"]
            for record in failed_records
        }

        pending_count = max(
            0,
            total_original_count
            - len(annotated_ids)
            - len(failed_ids)
        )

        requested_total = sum(
            stat["requested"]
            for stat in adapter_stats
        )

        selected_total = sum(
            stat["selected"]
            for stat in adapter_stats
        )

        shortfall_total = sum(
            stat["shortfall"]
            for stat in adapter_stats
        )

        source_breakdown = {}

        for stat in adapter_stats:

            source_breakdown[
                stat["source"]
            ] = {
                "requested": stat["requested"],
                "available": stat.get(
                    "available",
                    0
                ),
                "valid": stat.get(
                    "valid",
                    0
                ),
                "selected": stat["selected"],
                "shortfall": stat["shortfall"],
                "status": stat.get(
                    "status",
                    "unknown"
                )
            }

        annotation_coverage = (
            (
                len(annotated_records)
                / total_original_count
            )
            if total_original_count
            else 0.0
        )

        failure_rate = (
            (
                len(failed_records)
                / (
                    len(annotated_records)
                    + len(failed_records)
                )
            )
            if (
                len(annotated_records)
                + len(failed_records)
            )
            else 0.0
        )

        report = {
            "pipeline_version": (
                config.pipeline_version
            ),

            "timestamp": datetime.now(
                timezone.utc
            ).isoformat(),

            "random_seed": (
                config.random_seed
            ),

            "dataset_integrity": {
                "original_dataset_schema": [
                    "raw_text",
                    "source"
                ],
                "original_records": (
                    total_original_count
                ),
                "annotated_records": (
                    len(annotated_records)
                ),
                "failed_records": (
                    len(failed_records)
                ),
                "pending_records": (
                    pending_count
                ),
                "duplicate_original_padding": False,
                "fake_execution_results": False
            },

            "source_targets": {
                "total_requested": (
                    requested_total
                ),
                "total_selected": (
                    selected_total
                ),
                "total_shortfall": (
                    shortfall_total
                )
            },

            "source_breakdown": (
                source_breakdown
            ),

            "annotation_metrics": {
                "true_llm_annotated_count": (
                    len(annotated_records)
                ),
                "failed_annotation_count": (
                    len(failed_records)
                ),
                "pending_unannotated_count": (
                    pending_count
                ),
                "annotation_coverage": (
                    round(
                        annotation_coverage,
                        6
                    )
                ),
                "failure_rate": (
                    round(
                        failure_rate,
                        6
                    )
                ),
                "zero_fake_annotations_verified": True
            },

            "split_counts": split_counts,

            "domain_distribution": (
                domain_counts
            ),

            "category_distribution": (
                category_counts
            ),

            "intent_distribution": (
                intent_counts
            ),

            "source_distribution": (
                source_counts
            ),

            "llm_usage": {
                "provider": (
                    config.llm_provider
                ),
                "annotation_model": (
                    config.annotation_model
                ),
                "production_requests_used": (
                    ledger.production_requests_used
                ),
                "production_requests_remaining": (
                    ledger.remaining_requests()
                ),
                "cache_hits": (
                    ledger.cache_hits
                ),
                "retry_count": (
                    ledger.retries
                )
            }
        }

        report_path = os.path.join(
            config.output_dir,
            "dataset_report.json"
        )

        CheckpointManager.atomic_write_json(
            report_path,
            report
        )

        print(
            "Exported dataset_report.json successfully."
        )

        print()
        print(
            "========== DATASET QUALITY SUMMARY =========="
        )

        print(
            f"Original records : "
            f"{total_original_count:,}"
        )

        print(
            f"Annotated       : "
            f"{len(annotated_records):,}"
        )

        print(
            f"Failed          : "
            f"{len(failed_records):,}"
        )

        print(
            f"Pending         : "
            f"{pending_count:,}"
        )

        print(
            f"Annotation      : "
            f"{annotation_coverage:.2%}"
        )

        print(
            f"Requests used   : "
            f"{ledger.production_requests_used:,}"
        )

        print(
            f"Requests left   : "
            f"{ledger.remaining_requests():,}"
        )

        print(
            "=============================================="
        )

        return report


report = QualityReporter.generate_report(
    config=config,
    adapter_stats=adapter_stats,
    total_original_count=len(df_original),
    annotated_records=annotated_records,
    failed_records=failed_records,
    train_records=train_records,
    val_records=val_records,
    test_records=test_records,
    ledger=ledger
)

## 21. Final Verification
Comprehensive programmatic verification across all exported files, schemas, targets, and leakage assertions.

In [ ]:
# 21. Programmatic Verification & Colab Download Helper

print("=" * 80)
print("RUNNING FINAL PROGRAMMATIC VERIFICATION")
print("=" * 80)

verification_errors = []
verification_warnings = []


def verify_file_exists(path: str, label: str):
    if not os.path.exists(path):
        verification_errors.append(
            f"{label} missing: {path}"
        )
        return False

    return True


def verify_jsonl_file(
    path: str,
    expected_count: Optional[int] = None
):
    if not verify_file_exists(
        path,
        os.path.basename(path)
    ):
        return 0

    count = 0

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            for line_number, line in enumerate(
                f,
                start=1
            ):

                line = line.strip()

                if not line:
                    verification_errors.append(
                        f"{os.path.basename(path)} "
                        f"contains an empty line at "
                        f"{line_number}."
                    )
                    continue

                try:
                    record = json.loads(
                        line
                    )
                except Exception as e:
                    verification_errors.append(
                        f"{os.path.basename(path)} "
                        f"contains invalid JSON at line "
                        f"{line_number}: {e}"
                    )
                    continue

                if not isinstance(
                    record,
                    dict
                ):
                    verification_errors.append(
                        f"{os.path.basename(path)} "
                        f"line {line_number} is not a JSON object."
                    )
                    continue

                if "messages" not in record:
                    verification_errors.append(
                        f"{os.path.basename(path)} "
                        f"line {line_number} missing 'messages'."
                    )

                if "tools" not in record:
                    verification_errors.append(
                        f"{os.path.basename(path)} "
                        f"line {line_number} missing 'tools'."
                    )

                messages = record.get(
                    "messages",
                    []
                )

                if not isinstance(
                    messages,
                    list
                ):
                    verification_errors.append(
                        f"{os.path.basename(path)} "
                        f"line {line_number}: "
                        "'messages' must be a list."
                    )

                else:

                    roles = [
                        message.get("role")
                        for message in messages
                        if isinstance(message, dict)
                    ]

                    if "system" not in roles:
                        verification_errors.append(
                            f"{os.path.basename(path)} "
                            f"line {line_number}: "
                            "missing system message."
                        )

                    if "user" not in roles:
                        verification_errors.append(
                            f"{os.path.basename(path)} "
                            f"line {line_number}: "
                            "missing user message."
                        )

                tools = record.get(
                    "tools",
                    []
                )

                if not isinstance(
                    tools,
                    list
                ):
                    verification_errors.append(
                        f"{os.path.basename(path)} "
                        f"line {line_number}: "
                        "'tools' must be a list."
                    )

                count += 1

    except Exception as e:

        verification_errors.append(
            f"Could not read "
            f"{os.path.basename(path)}: {e}"
        )

        return count

    if (
        expected_count is not None
        and count != expected_count
    ):
        verification_errors.append(
            f"{os.path.basename(path)} expected "
            f"{expected_count:,} records but found "
            f"{count:,}."
        )

    return count


# ============================================================
# 1. Verify original_dataset.csv
# ============================================================

orig_csv = os.path.join(
    config.output_dir,
    "original_dataset.csv"
)

if verify_file_exists(
    orig_csv,
    "original_dataset.csv"
):

    try:

        df_orig_chk = pd.read_csv(
            orig_csv
        )

        expected_columns = [
            "raw_text",
            "source"
        ]

        if list(
            df_orig_chk.columns
        ) != expected_columns:

            verification_errors.append(
                "original_dataset.csv must contain "
                "exactly ['raw_text', 'source']."
            )

        if df_orig_chk.empty:

            verification_errors.append(
                "original_dataset.csv is empty."
            )

        if (
            df_orig_chk["raw_text"]
            .isna()
            .any()
        ):

            verification_errors.append(
                "Null raw_text found."
            )

        if (
            df_orig_chk["source"]
            .isna()
            .any()
        ):

            verification_errors.append(
                "Null source found."
            )

        if (
            df_orig_chk["raw_text"]
            .astype(str)
            .str.strip()
            .eq("")
            .any()
        ):

            verification_errors.append(
                "Empty raw_text found."
            )

        # Verify no duplicate source/text pair.
        duplicate_count = (
            df_orig_chk
            .duplicated(
                subset=[
                    "raw_text",
                    "source"
                ]
            )
            .sum()
        )

        if duplicate_count > 0:

            verification_errors.append(
                f"Found {duplicate_count:,} duplicate "
                "raw_text + source pairs."
            )

        actual_original_count = len(
            df_orig_chk
        )

        requested_target = (
            config.total_target
        )

        if actual_original_count < requested_target:

            shortfall = (
                requested_target
                - actual_original_count
            )

            verification_warnings.append(
                "Original dataset is below the requested "
                f"target by {shortfall:,} records. "
                "No duplicate padding was performed."
            )

        elif actual_original_count > requested_target:

            verification_warnings.append(
                "Original dataset contains more records "
                f"than the configured target "
                f"({requested_target:,})."
            )

        print(
            "[PASSED] original_dataset.csv:"
        )

        print(
            f"        Rows: {actual_original_count:,}"
        )

        print(
            "        Schema: raw_text + source"
        )

        print(
            f"        Requested target: "
            f"{requested_target:,}"
        )

        print(
            f"        Shortfall: "
            f"{max(0, requested_target - actual_original_count):,}"
        )

    except Exception as e:

        verification_errors.append(
            f"Failed reading original_dataset.csv: {e}"
        )


# ============================================================
# 2. Verify failed_annotations.csv
# ============================================================

failed_csv = os.path.join(
    config.output_dir,
    "failed_annotations.csv"
)

if verify_file_exists(
    failed_csv,
    "failed_annotations.csv"
):

    try:

        df_failed_chk = pd.read_csv(
            failed_csv
        )

        expected_failed_columns = [
            "record_id",
            "source",
            "raw_text",
            "error_category",
            "error_message",
            "raw_llm_response",
            "timestamp"
        ]

        if list(
            df_failed_chk.columns
        ) != expected_failed_columns:

            verification_errors.append(
                "failed_annotations.csv has "
                "an unexpected schema."
            )

        print(
            "[PASSED] failed_annotations.csv: "
            f"{len(df_failed_chk):,} failed records isolated."
        )

    except Exception as e:

        verification_errors.append(
            f"Failed reading failed_annotations.csv: {e}"
        )


# ============================================================
# 3. Verify synthetic_dataset.csv
# ============================================================

synth_csv = os.path.join(
    config.output_dir,
    "synthetic_dataset.csv"
)

if verify_file_exists(
    synth_csv,
    "synthetic_dataset.csv"
):

    try:

        df_synth_chk = pd.read_csv(
            synth_csv
        )

        expected_synth_columns = [
            "record_id",
            "source",
            "raw_text",
            "domain",
            "category",
            "goal",
            "intents",
            "entities",
            "missing_information",
            "requires_confirmation",
            "execution_plan",
            "tool_calls",
            "expected_response",
            "model"
        ]

        if list(
            df_synth_chk.columns
        ) != expected_synth_columns:

            verification_errors.append(
                "synthetic_dataset.csv has "
                "an unexpected schema."
            )

        print(
            "[PASSED] synthetic_dataset.csv: "
            f"{len(df_synth_chk):,} annotated records."
        )

    except Exception as e:

        verification_errors.append(
            f"Failed reading synthetic_dataset.csv: {e}"
        )


# ============================================================
# 4. Verify JSONL files
# ============================================================

jsonl_expectations = {
    "latentspace_complete_dataset.jsonl": (
        len(annotated_records)
    ),
    "latentspace_train.jsonl": (
        len(train_records)
    ),
    "latentspace_validation.jsonl": (
        len(val_records)
    ),
    "latentspace_test.jsonl": (
        len(test_records)
    )
}

jsonl_counts = {}

for filename, expected_count in (
    jsonl_expectations.items()
):

    fpath = os.path.join(
        config.output_dir,
        filename
    )

    actual_count = verify_jsonl_file(
        fpath,
        expected_count
    )

    jsonl_counts[
        filename
    ] = actual_count

    if actual_count == expected_count:
        print(
            f"[PASSED] {filename}: "
            f"{actual_count:,} valid JSONL records."
        )


# ============================================================
# 5. Verify dataset_report.json
# ============================================================

report_path = os.path.join(
    config.output_dir,
    "dataset_report.json"
)

if verify_file_exists(
    report_path,
    "dataset_report.json"
):

    try:

        with open(
            report_path,
            "r",
            encoding="utf-8"
        ) as f:

            report_check = json.load(
                f
            )

        if not isinstance(
            report_check,
            dict
        ):
            verification_errors.append(
                "dataset_report.json is not a JSON object."
            )

        else:

            required_report_sections = [
                "pipeline_version",
                "timestamp",
                "dataset_integrity",
                "source_targets",
                "annotation_metrics",
                "split_counts",
                "llm_usage"
            ]

            missing_sections = [
                section
                for section in required_report_sections
                if section not in report_check
            ]

            if missing_sections:

                verification_errors.append(
                    "dataset_report.json missing sections: "
                    + ", ".join(missing_sections)
                )

            else:

                print(
                    "[PASSED] dataset_report.json: "
                    "valid JSON and required sections present."
                )

    except Exception as e:

        verification_errors.append(
            f"Failed reading dataset_report.json: {e}"
        )


# ============================================================
# 6. Cross-check in-memory counts
# ============================================================

if (
    "actual_original_count" in locals()
):

    expected_pending = max(
        0,
        actual_original_count
        - len(annotated_records)
        - len(failed_records)
    )

    if report_check.get(
        "annotation_metrics",
        {}
    ).get(
        "pending_unannotated_count"
    ) != expected_pending:

        verification_errors.append(
            "dataset_report pending count does not "
            "match calculated pending count."
        )


# ============================================================
# 7. Verify split accounting
# ============================================================

split_total = (
    len(train_records)
    + len(val_records)
    + len(test_records)
)

if split_total != len(
    annotated_records
):

    verification_errors.append(
        "Train + validation + test counts do not "
        "equal the accepted annotation count."
    )

else:

    print(
        "[PASSED] Split accounting:"
        f" {split_total:,} = "
        f"{len(annotated_records):,} annotated records."
    )


# ============================================================
# 8. Verify no fake execution results
# ============================================================

fake_result_patterns = [
    '"status": "success"',
    '"status":"success"',
    '"execution_status": "success"',
    '"execution_status":"success"',
    '"result": "success"',
    '"result":"success"'
]

fake_execution_detected = False

for record in annotated_records:

    serialized = json.dumps(
        record,
        ensure_ascii=False
    ).lower()

    for pattern in fake_result_patterns:

        if pattern.lower() in serialized:

            fake_execution_detected = True

            verification_errors.append(
                "Possible fabricated execution result "
                f"detected in record "
                f"{record.get('record_id')}."
            )

            break

if not fake_execution_detected:

    print(
        "[PASSED] No known fabricated execution-result "
        "patterns detected."
    )


# ============================================================
# 9. Final Verification Result
# ============================================================

print()
print("=" * 80)

if verification_warnings:

    print(
        "VERIFICATION WARNINGS"
    )

    for warning in verification_warnings:

        print(
            f"[WARNING] {warning}"
        )

    print()


if verification_errors:

    print(
        "FINAL VERIFICATION FAILED"
    )

    print()

    for error in verification_errors:

        print(
            f"[FAILED] {error}"
        )

    print()
    print("=" * 80)

    raise AssertionError(
        f"Final verification failed with "
        f"{len(verification_errors)} issue(s)."
    )

else:

    print(
        "ALL PROGRAMMATIC VERIFICATION CHECKS PASSED!"
    )

    print(
        f"Original records verified: "
        f"{len(df_orig_chk):,}"
    )

    print(
        f"Accepted LLM annotations: "
        f"{len(annotated_records):,}"
    )

    print(
        f"Failed annotations isolated: "
        f"{len(failed_records):,}"
    )

    print(
        f"Train / Validation / Test: "
        f"{len(train_records):,} / "
        f"{len(val_records):,} / "
        f"{len(test_records):,}"
    )

print("=" * 80)


# ============================================================
# 10. Optional Google Colab Download Helper
# ============================================================

try:

    from google.colab import files

    print()
    print(
        "Google Colab detected."
    )

    print(
        "To download an output manually, run:"
    )

    print(
        "# files.download("
        "'latentspace_dataset/output/"
        "original_dataset.csv'"
        ")"
    )

    print(
        "# files.download("
        "'latentspace_dataset/output/"
        "latentspace_complete_dataset.jsonl'"
        ")"
    )

except ImportError:

    pass